In [18]:
import boto3
from dotenv import load_dotenv
import os
import io
from PIL import Image
import base64
from tqdm import tqdm

load_dotenv("../../backend/.env")

session = boto3.Session(
    aws_access_key_id=os.getenv('ACCESS_KEY'),
    aws_secret_access_key=os.getenv('SECRET_ACCESS_KEY'),
    aws_session_token=os.getenv('SESSION_TOKEN')
)

s3 = session.resource('s3')
path = "data/equation/"
bucket = s3.Bucket("penman-lln")

objs = bucket.objects.filter(Prefix=path)

In [ ]:
signature = b"\x89PNG"

for obj in tqdm(objs):
    if obj.key == path:
        continue
        
    curr_url = s3.Object("penman-lln", obj.key).get()["Body"].read()

    if curr_url.startswith(signature):
        continue

    image = Image.open(io.BytesIO(base64.decodebytes(curr_url)))

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    buffer.seek(0)

    s3_obj = s3.Object("penman-lln", obj.key)
    s3_obj.put(
        Body=buffer,
        ContentType="image/png"
    )